# EDA — Prédiction HPP sévère (synthèse)

Parcours : `00_Prepare_Data` → **`01_EDA`** → `02_Preprocessing` → `03_Model_Comparison`.

**Objectif** : prédire `hpp_trans` **avant l'accouchement** (anti leakage).  
**Priorité métier** : maximiser le **rappel**.

Données : `processed/hpp_prepartum.csv` si dispo, sinon extrait portfolio — voir `../00_Data/DATA_LOCATION.md`.

## 1. Chargement

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from hpp_data import FEATURE_ORDER, TARGET, DICO_CSV, load_hpp

df, source = load_hpp()
print(f"Source : {source} — {df.shape[0]} lignes × {df.shape[1]} colonnes")
if TARGET in df.columns:
    print(df[TARGET].value_counts(normalize=True).rename("part").round(4))
desc = pd.read_csv(DICO_CSV, index_col=0) if DICO_CSV.exists() else None
df.head(3)

## 2. Périmètre pré-accouchement

Features alignées sur le modèle déployé (API / Streamlit).

In [ ]:
available = [c for c in FEATURE_ORDER if c in df.columns]
missing_cols = [c for c in FEATURE_ORDER if c not in df.columns]
print("Features :", len(available), "/", len(FEATURE_ORDER))
if missing_cols:
    print("Absentes :", missing_cols)

use_cols = available + ([TARGET] if TARGET in df.columns else [])
eda = df[use_cols].copy()
eda.head(3)

## 3. Qualité des données (résumé)

In [ ]:
quality = pd.DataFrame({
    "dtype": eda.dtypes.astype(str),
    "n_missing": eda.isna().sum(),
    "pct_missing": (eda.isna().mean() * 100).round(2),
    "n_unique": eda.nunique(dropna=True),
}).sort_values("pct_missing", ascending=False)
quality

## 4. Cible `hpp_trans` — déséquilibre

≈ 2 % de positifs en base complète → SMOTE / class_weight testés en `03`.

In [ ]:
if TARGET in eda.columns:
    counts = eda[TARGET].value_counts()
    fig, ax = plt.subplots(figsize=(5, 3))
    counts.plot(kind="bar", ax=ax, color=["#1B2A4A", "#B03A2E"])
    ax.set_title(f"Distribution hpp_trans ({source})")
    ax.set_xlabel("Classe")
    ax.set_ylabel("Effectif")
    plt.tight_layout()
    plt.show()
    print((counts / counts.sum() * 100).round(2).astype(str) + " %")
else:
    print("Colonne cible absente")

## 5. Décisions de features (EDA exploratoire)

| Écartée | Motif |
|---------|--------|
| `type_grossesse` | Colinéaire avec `g_type` |
| `parite_cor`, `bas_risque` | Peu informatives |
| `Aide_procreation` | Colinéaire avec `AMP` |
| `Dosecortico` | Trop de NA vs `cortico` |
| `sej18` | Colinéaire avec `nsej18` |
| `poids_mere` | Colinéaire avec `bmi` |
| `hta_gest`, `hta_chro` | Redondantes vs `hta_tot` |

Nettoyage : `dsm_g` ∈ `[0, 270)` ; NA `age_m` droppés — appliqués dans `00_Prepare_Data`.

Détail → `_archive/01_EDA_exploration.ipynb`.

## 6. Suite

1. `02_Preprocessing.ipynb` — dropna + ColumnTransformer  
2. `03_Model_Comparison.ipynb` — modèles + **MLflow**  
3. Artefact : `../app/model/artifacts/model.joblib`